# Part b: Regression on the Runge function

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # add Code/ so `src` can be imported

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from src.data import runge, make_runge_split, design_matrix, scale_data, center_data
from src.regression import *
from src.functions import *
from src.plotting import *
from src.network import NeuralNetwork 
from src.optimizers import Adam 

In [ ]:
#Recreate dataset from project 1 
X_train, X_test, y_train, y_test = make_runge_split(
    n = 100,
    sigma = 0.1,
    test_size = 0.2,
    seed = 2026
)

# Design matrix of degree 8 (optimal degree from project 1) 
Xt_train = design_matrix(X_train, degree=8, intercept=False)
Xt_test = design_matrix(X_test, degree=8, intercept=False)

# Center and scale the data 
Xt_train, Xt_test, X_mean, X_std = scale_data(Xt_train, Xt_test )
y_train_c, y_test_c, y_mean = center_data(y_train, y_test)

#Fitting the ols 
theta = ols_fit(Xt_train, y_train_c)

y_pred_ols = Xt_test @ theta + y_mean 
mse_ols = mse(y_pred_ols, y_test)

print(f"OLS test MSE: {mse_ols:.6f}")

# Generate x-values for a smooth curve
x_plot = np.linspace(-1, 1, 300).reshape(-1, 1)

# Use the same transformation as for the training data
Xt_plot = design_matrix(x_plot, degree=8, intercept=False)
Xt_plot = (Xt_plot - X_mean) / X_std

# Predict using the fitted OLS model
y_ols_plot = Xt_plot @ theta + y_mean

# True Runge function
y_true = runge(x_plot)

fig, ax = line_plot(
    x_plot,
    {
        "True Runge function": y_true,
        "OLS, degree 8": y_ols_plot
    },
    xlabel="$x$",
    ylabel="$y$",
    data=(X_train, y_train),
    name="b_ols_fit")


Notes on the plot:
- noise is 0.1, variance of this is 0.01, all models would get mse with minimum 0.01. 
- oscillations on edges is runge phenomenon. 
- peak is not as sharp because of degree 8. 

## Using the network 

### Network with 50 nodes 

In [ ]:
# scale the original data 
X_train_s, X_test_s, x_mean, x_std = scale_data(X_train, X_test)

net50 = NeuralNetwork(
    network_input_size=1,
    layer_output_sizes=[50, 1],
    activation_funcs=[sigmoid, identity],
    activation_ders=[sigmoid_der, identity_der],
    cost_fun=mse,
    cost_der=mse_der,
    optimizer=Adam(gamma=0.01),
    seed=2026)

history50 = net50.train(
    X_train_s, 
    y_train, 
    epochs=500,
    batch_size=14,
    rng=np.random.default_rng(2026))

yn50_plot = net50.predict((x_plot - x_mean)/x_std) #scaling 

fig, ax = line_plot(
    x_plot,
    {
        "True Runge function": y_true,
        "OLS, degree 8": y_ols_plot,
        "NN, 1×50 sigmoid": yn1_plot,
    },
    xlabel="$x$",
    ylabel="$y$",
    data=(X_train, y_train),
    name="b50_nn_vs_ols")


Notes on the plot:
- nn removes runge phenomenon -> not polynomial basis anymore 
- shaper peak
- drops at right edge?? (fewer training points? )

In [ ]:
epochs = np.arange(1, len(history50) + 1)
fig, ax = line_plot(epochs, {"Adam": history50},
                    xlabel="Epoch", ylabel="Training MSE",
                    name="b50_learning_curve")

Notes on the plot:
- Loss drops fast -> only need around 100 epoch for future runs
- levels around 0.02, close to noise floor. 

In [ ]:
y_pred_n50 = net50.predict(X_test_s)
mse_n50 = mse(y_pred_n50, y_test)
print(f"NN50 test MSE: {mse_n50:.6f}")

Notes on mse:
- few test points -> use different seed for final comparison 
- difference to ols is more emaningful when taking into consideration the noise. 

### Network with 100 nodes 

In [ ]:
net100 = NeuralNetwork(
    network_input_size=1,
    layer_output_sizes=[100, 1],
    activation_funcs=[sigmoid, identity],
    activation_ders=[sigmoid_der, identity_der],
    cost_fun=mse,
    cost_der=mse_der,
    optimizer=Adam(gamma=0.01),
    seed=2026)

history100 = net100.train(
    X_train_s, 
    y_train, 
    epochs=500,
    batch_size=14,
    rng=np.random.default_rng(2026))

yn100_plot = net100.predict((x_plot - x_mean)/x_std) #scaling 

fig, ax = line_plot(
    x_plot,
    {
        "True Runge function": y_true,
        "OLS, degree 8": y_ols_plot,
        "NN, 1×100 sigmoid": yn100_plot,
    },
    xlabel="$x$",
    ylabel="$y$",
    data=(X_train, y_train),
    name="b100_nn_vs_ols")

y_pred_n100 = net100.predict(X_test_s)
mse_n100 = mse(y_pred_n100, y_test)
print(f"NN100 test MSE: {mse_n100:.6f}")

epochs = np.arange(1, len(history100) + 1)
fig, ax = line_plot(epochs, {"Adam": history100},
                    xlabel="Epoch", ylabel="Training MSE",
                    name="b100_learning_curve")

Notes on plots and mse:
- mse is higher ? 
- nn now has a worse fit ? 
- leanring curve is way more spikey (too big learning rate?) 


In [ ]:
structure = {
    "2×50":  [50, 50, 1],
    "2×100": [100, 100, 1]}

results = {}
for name, sizes in structure.items():
    nodes = len(sizes) - 1
    net = NeuralNetwork(
        network_input_size=1,
        layer_output_sizes=sizes,
        activation_funcs=[sigmoid] * nodes + [identity],
        activation_ders=[sigmoid_der] * nodes + [identity_der],
        cost_fun=mse,
        cost_der=mse_der,
        optimizer=Adam(gamma=0.01),     # new optimizer for every network
        seed=2026)
    history = net.train(X_train_s, y_train, epochs=500, batch_size=14,
                        rng=np.random.default_rng(2026))
    results[name] = {
        "test_mse": mse(net.predict(X_test_s), y_test),
        "history": history,
        "plot": net.predict((x_plot - x_mean) / x_std)}


fig, ax = line_plot(
    x_plot,
    {
        "True Runge function": y_true,
        "OLS, degree 8": y_ols_plot,
        "NN 2×50":  results["2×50"]["plot"],
    },
    xlabel="$x$",
    ylabel="$y$",
    data=(X_train, y_train),
    name="b_2x500_fit")

fig, ax = line_plot(
    x_plot,
    {
        "True Runge function": y_true,
        "OLS, degree 8": y_ols_plot,
        "NN 2×100": results["2×100"]["plot"],
    },
    xlabel="$x$",
    ylabel="$y$",
    data=(X_train, y_train),
    name="b_2x100_fit")

# Test MSE for the two-layer networks
for name, res in results.items():
    print(f"NN {name} test MSE: {res['test_mse']:.6f}")

# Learning curves for both networks in one plot
epochs = np.arange(1, len(results["2×50"]["history"]) + 1)
fig, ax = line_plot(
    epochs,
    {
        "NN 2×50":  results["2×50"]["history"],
        "NN 2×100": results["2×100"]["history"],
    },
    xlabel="Epoch",
    ylabel="Training MSE",
    name="b_2layers_learning_curve")

In [ ]:
import pandas as pd

sigma = 0.1
noise_floor = sigma**2

table = {
    "OLS, degree 8": mse_ols,
    "NN 1×50":       mse_n50,
    "NN 1×100":      mse_n100,
    "NN 2×50":       results["2×50"]["test_mse"],
    "NN 2×100":      results["2×100"]["test_mse"]}
table = {model: float(m) for model, m in table.items()}

df = pd.DataFrame({
    "Test MSE": table.values(),
    "Above noise floor": [m - noise_floor for m in table.values()],
}, index=table.keys())

display(df.round(4))
print(df.to_latex(float_format="%.4f"))